Imports

In [1]:
# ============================================================
# Window-to-Wall Ratio Segmentation Framework
# Intelligent Computing Version
# ============================================================

import os
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

Environment Configuration

In [2]:
# ============================================================
# Environment Configuration
# ============================================================

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE

GPU Configuration

In [3]:
# ============================================================
# GPU Configuration
# ============================================================

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("TensorFlow :", tf.__version__)
print("GPU :", tf.config.list_physical_devices("GPU"))

TensorFlow : 2.20.0
GPU : []


Mixed Precision

In [4]:
# ============================================================
# Mixed Precision
# ============================================================

from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy("mixed_float16")

print("Mixed Precision :", mixed_precision.global_policy())

Mixed Precision : <DTypePolicy "mixed_float16">


Mount Google Drive

In [5]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


Copy Dataset to Local SSD

In [6]:
# ============================================================
# Copy Dataset
# ============================================================

!cp "/content/drive/MyDrive/WWR_Seg_Model/data.zip" /content/

!unzip -oq /content/data.zip -d /content/

Configuration

In [7]:
@dataclass
class Config:

    IMAGE_SIZE = 256

    IMAGE_CHANNELS = 3

    MASK_CHANNELS = 1

    NUM_CLASSES = 4

    BATCH_SIZE = 16

    EPOCHS = 100

    LEARNING_RATE = 1e-4

    WEIGHT_DECAY = 1e-5

    BUFFER_SIZE = 1000

    SEED = 42

    TRAIN_RATIO = 0.90

    VAL_RATIO = 0.10

Segmentation Classes

In [ ]:
# ============================================================
# Segmentation Classes
# ============================================================

CLASS_INFO = {
    0: {"name": "Roof",   "gray": 70},
    1: {"name": "Window", "gray": 129},
    2: {"name": "Wall",   "gray": 221},
    3: {"name": "Other",  "gray": 255},
}

CLASS_NAMES = [v["name"] for v in CLASS_INFO.values()]

GRAY_VALUES = [v["gray"] for v in CLASS_INFO.values()]

GRAY_TO_CLASS = {
    70: 0,
    129: 1,
    221: 2,
    255: 3
}

CLASS_TO_GRAY = {
    0: 70,
    1: 129,
    2: 221,
    3: 255
}

print("="*50)

print("Segmentation Classes")

print("="*50)

for idx in CLASS_INFO:

    print(
        f"{idx} -> "
        f"{CLASS_INFO[idx]['name']} "
        f"(Gray={CLASS_INFO[idx]['gray']})"
    )

Project Paths

In [8]:
# ============================================================
# Project Paths
# ============================================================

# ---------- Local SSD (Fast Dataset) ----------
LOCAL_ROOT = Path("/content")

IMAGE_DIR = LOCAL_ROOT / "images"
MASK_DIR  = LOCAL_ROOT / "masks"

# ---------- Google Drive (Persistent Storage) ----------
DRIVE_ROOT = Path("/content/drive/MyDrive/WWR_Seg_Model")

MODEL_DIR      = DRIVE_ROOT / "models"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
LOG_DIR        = DRIVE_ROOT / "logs"
RESULT_DIR     = DRIVE_ROOT / "results"

# ---------- Create folders if they do not exist ----------
for folder in [MODEL_DIR, CHECKPOINT_DIR, LOG_DIR, RESULT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Dataset Directory :", LOCAL_ROOT)
print("Project Directory :", DRIVE_ROOT)

Verify Dataset

In [9]:
# ============================================================
# Verify Dataset
# ============================================================

assert IMAGE_DIR.exists(), f"Missing folder: {IMAGE_DIR}"
assert MASK_DIR.exists(), f"Missing folder: {MASK_DIR}"

image_files = sorted(IMAGE_DIR.glob("*.png"))
mask_files  = sorted(MASK_DIR.glob("*.png"))

print(f"Images : {len(image_files)}")
print(f"Masks  : {len(mask_files)}")

assert len(image_files) == len(mask_files), "Image/Mask count mismatch!"

# Check filenames
image_names = [x.name for x in image_files]
mask_names  = [x.name for x in mask_files]

assert image_names == mask_names, "Image and Mask filenames do not match."

print("✓ Dataset structure verified.")

Images : 1000
Masks  : 1000
Dataset verified successfully.


Project Summary

In [10]:
# ============================================================
# Project Summary
# ============================================================

print("="*60)

print("Window Segmentation Framework")

print("="*60)

print("Image Size :", Config.IMAGE_SIZE)

print("Batch Size :", Config.BATCH_SIZE)

print("Epochs     :", Config.EPOCHS)

print("Classes    :", Config.NUM_CLASSES)

print("Dataset    :", len(image_files))

print("="*60)

Window Segmentation Framework
Image Size : 256
Batch Size : 16
Epochs     : 100
Classes    : 4
Dataset    : 1000
